# PubMed Relevance Classifier

This notebook recreates the workflow from [Pubmed-Pipeline](https://github.com/nicford/Pubmed-Pipeline) in a **pandas + scikit-learn** stack (no Spark).

1. Download papers from **NCBI PubMed** (E-utilities)
2. Parse Medline XML into a table
3. Train a text classifier (`Relevant` vs `Not Relevant`)
4. Run a setup pipeline that keeps only relevant papers and saves **Parquet**

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Kaggle: clone or upload this folder; local: run from repo root
ROOT = Path("/kaggle/working/pubmed") if Path("/kaggle/working").exists() else Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.download import count_pubmed_results
from src.model import RELEVANT_LABEL, NOT_RELEVANT_LABEL, save_classifier, train_classifier
from src.parse_papers import parse_xml_directory, preprocess_dataframe
from src.pipeline import PubmedPipelineSetup
from src.download import download_pubmed_xml

ARTIFACTS = ROOT / "artifacts"
DATA = ROOT / "data"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

API_KEY = None  # set your NCBI API key string for faster downloads
POSITIVE_QUERY = ["machine learning[Title/Abstract] AND oncology[MeSH Terms]"]
NEGATIVE_QUERY = ["cardiology[MeSH Terms] NOT machine learning[Title/Abstract]"]
SETUP_QUERIES = POSITIVE_QUERY
MAX_RECORDS = 300  # increase for production; keep small for notebook demos

## 1. Check PubMed result counts

In [ ]:
print("Positive matches:", count_pubmed_results(POSITIVE_QUERY, API_KEY))
print("Negative matches:", count_pubmed_results(NEGATIVE_QUERY, API_KEY))

## 2. Train classifier (weak labels from two queries)

Papers from the **positive** query are labeled `Relevant`; papers from the **negative** query are `Not Relevant`.
For production, use hand-labeled `labels.csv` instead (see README).

In [ ]:
pos_xml = DATA / "train_positive_xml"
neg_xml = DATA / "train_negative_xml"

download_pubmed_xml(pos_xml, POSITIVE_QUERY, API_KEY, max_records=MAX_RECORDS)
download_pubmed_xml(neg_xml, NEGATIVE_QUERY, API_KEY, max_records=MAX_RECORDS)

pos_df = preprocess_dataframe(parse_xml_directory(pos_xml))
neg_df = preprocess_dataframe(parse_xml_directory(neg_xml))

pos_df["label"] = RELEVANT_LABEL
neg_df["label"] = NOT_RELEVANT_LABEL

train_df = pd.concat([pos_df, neg_df], ignore_index=True).drop_duplicates(subset=["pmid"])
pipeline, metrics = train_classifier(train_df, train_df["label"])

classifier_path = ARTIFACTS / "classifier.joblib"
save_classifier(pipeline, classifier_path)
print(metrics["report"])

## 3. Setup pipeline — download, classify, save Parquet

In [ ]:
setup = PubmedPipelineSetup(
    xml_dir=DATA / "xml",
    classifier_path=classifier_path,
    dataframe_output_path=DATA / "relevant_papers.parquet",
    last_run_path=ARTIFACTS / "last_run.pkl",
)

setup.download_xml_from_pubmed(SETUP_QUERIES, API_KEY, max_records=MAX_RECORDS)
relevant_df = setup.run_pipeline()
relevant_df.head()

## 4. Inspect output

In [ ]:
print(f"Relevant papers saved: {len(relevant_df)}")
relevant_df[["pmid", "title", "pubdate"]].head(10)